# ABSA Training
Aspect-Based Sentiment Analysis on Arabic reviews using `aubmindlab/bert-base-arabertv2`

## 1. Imports

In [1]:
import re
import json
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

PyTorch version : 2.11.0+cpu
CUDA available  : False


## 2. Load Data

In [2]:
df = pd.read_excel("DeepX_train.xlsx")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (1971, 9)
Columns: ['review_id', 'review_text', 'star_rating', 'date', 'business_name', 'business_category', 'platform', 'aspects', 'aspect_sentiments']


,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


## 3. Parse JSON Columns

In [3]:
df["aspects"] = df["aspects"].apply(json.loads)
df["aspect_sentiments"] = df["aspect_sentiments"].apply(json.loads)
print("JSON columns parsed successfully.")

JSON columns parsed successfully.


## 4. Explode to One Row per Aspect

In [4]:
rows = []
for _, row in df.iterrows():
    review = row["review_text"]
    for aspect, sentiment in row["aspect_sentiments"].items():
        rows.append({
            "text": review,
            "aspect": aspect,
            "label": sentiment
        })

train_df = pd.DataFrame(rows)
print(f"Total rows after explosion: {len(train_df)}")
train_df.head()

Total rows after explosion: 3333


,text,aspect,label
0,لا يوجد الدفع بالبطاقه عند الاستلام,app_experience,negative
1,لا يوجد الدفع بالبطاقه عند الاستلام,delivery,negative
2,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,cleanliness,positive
3,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,ambiance,positive
4,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,service,positive


## 5. Label Encoding

In [5]:
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}

train_df["label_id"] = train_df["label"].map(LABEL_MAP)

# Sanity check — catch any unmapped labels
assert train_df["label_id"].notna().all(), f"Unmapped labels found: {train_df[train_df['label_id'].isna()]['label'].unique()}"

print(train_df[["label", "label_id"]].value_counts().sort_index())

label     label_id
negative  0           1538
neutral   1            149
positive  2           1646
Name: count, dtype: int64


## 6. Text Cleaning

In [ ]:
def clean_text(text: str) -> str:
    text = str(text)

    # Remove newlines
    text = re.sub(r"\n+", " ", text)

    # Remove dates
    text = re.sub(r"\d{4}-\d{2}-\d{2}", "", text)
    text = re.sub(r"\d{2}/\d{2}/\d{4}", "", text)

    # Underscore → space
    text = text.replace("_", " ")

    # Remove special symbols
    text = re.sub(r"[@#$%^&*+!~|?><\-]", "", text)

    # Remove repeated dots
    text = re.sub(r"\.{2,}", "", text)

    # Remove non-Arabic, non-word characters
    text = re.sub(r"[^\w\s\u0600-\u06FF]", "", text)

    # Normalize Arabic letters
    text = re.sub(r"[أإآ]", "ا", text)
    text = re.sub(r"ة", "ه", text)

    # Collapse repeated characters (max 2)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    text = text.replace("٠","0").replace("١","1").replace("٢","2")\
               .replace("٣","3").replace("٤","4").replace("٥","5")\
               .replace("٦","6").replace("٧","7").replace("٨","8").replace("٩","9")

    # Clean whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning
train_df["text_clean"] = train_df["text"].apply(clean_text)

# Preview before vs after
print("Before:", train_df["text"].iloc[0][:100])
print("After :", train_df["text_clean"].iloc[0][:100])

Before: لا يوجد الدفع بالبطاقه عند الاستلام
After : لا يوجد الدفع بالبطاقه عند الاستلام


## 7. Build Model Input  
Format: `<review> [SEP] <aspect>` — the `[SEP]` token gives the model a clear boundary between review and aspect.

In [7]:
train_df["input"] = train_df["text_clean"] + " [SEP] " + train_df["aspect"]

print(f"Sample input:\n{train_df['input'].iloc[0]}")

Sample input:
لا يوجد الدفع بالبطاقه عند الاستلام [SEP] app_experience


## 8. Train / Validation Split

In [8]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label_id"]   # keep class balance in both splits
)

train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f"Train : {len(train_data)}")
print(f"Val   : {len(val_data)}")
print("\nTrain label distribution:")
print(train_data["label"].value_counts())

Train : 2666
Val   : 667

Train label distribution:
label
positive    1317
negative    1230
neutral      119
Name: count, dtype: int64


## 9. Load Tokenizer & Model

In [9]:
MODEL_NAME = "aubmindlab/bert-base-arabertv2"
NUM_LABELS = 3
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True
)

print("Model loaded successfully ✅")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully ✅
Parameters: 135,195,651


## 10. Dataset Class

In [10]:
class ABSADataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.texts     = df["input"].tolist()
        self.labels    = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(),
            "attention_mask" : encoding["attention_mask"].squeeze(),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_dataset = ABSADataset(train_data, tokenizer)
val_dataset   = ABSADataset(val_data,   tokenizer)

# Quick sanity check on a single sample
sample = train_dataset[0]
print(f"input_ids shape     : {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"label               : {sample['labels'].item()}")

input_ids shape     : torch.Size([128])
attention_mask shape: torch.Size([128])
label               : 0


In [11]:
!pip install langdetect

  DEPRECATION: Building 'langdetect' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'langdetect'. Discussion can be found at https://github.com/pypa/pip/issues/6334



     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ----------------------------- ------- 786.4/981.5 kB 10.0 MB/s eta 0:00:01
     -------------------------------------- 981.5/981.5 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993250 sha256=20949b6fa9436b068635f4d065770289b59cff822b01ca79115e4db31a094672
  Stored in directory: c:\users\yomna\appdata\local\pip\cache\wheels\eb\87\25\2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect


In [14]:
from langdetect import detect
from collections import Counter

langs = []

for text in train_df["text"]:
    try:
        langs.append(detect(str(text)))
    except:
        langs.append("unknown")

print(Counter(langs))

Counter({'ar': 3182, 'fa': 56, 'en': 39, 'ur': 33, 'so': 9, 'ro': 4, 'nl': 3, 'pl': 3, 'unknown': 2, 'it': 1, 'de': 1})
